# cAPTure: fold-aware feature profile

This CPU-only notebook profiles the completed canonical FULL_DEV Parquet artifacts before model preprocessing is frozen. It validates deterministic source/destination TCP-port roles, layered protocol indicators, numeric ranges, categorical code coverage, and fold-training constants. It does not refit anything on validation data, materialize transformed packets, construct graph windows, or train a model.


## 1. Mount Drive and load the project


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )

required_files = [
    PROJECT_ROOT / "code/python/utils/capture_feature_profile.py",
    PROJECT_ROOT / "code/python/tests/test_capture_feature_profile.py",
    PROJECT_ROOT / "configs/capture_experiment_v1.yaml",
    PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml",
    PROJECT_ROOT / "configs/capture_preprocessing_v1.yaml",
    PROJECT_ROOT / "code/python/requirements-capture.txt",
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Update the repository copy first: {missing_files}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))
print("CPU feature-profile environment is ready.")


## 2. Run synthetic contract checks


In [ ]:
test_environment = dict(os.environ)
test_environment["PYTHONPATH"] = str(PROJECT_ROOT / "code/python")
subprocess.run(
    [sys.executable, "-m", "unittest", "discover",
     "-s", str(PROJECT_ROOT / "code/python/tests"),
     "-p", "test_capture_feature_profile.py", "-v"],
    env=test_environment,
    cwd=PROJECT_ROOT,
    check=True,
)


## 3. Configure the FULL_DEV profile

The prepared run is immutable input. This profile writes only a small JSON report and configuration snapshots to a new Drive directory.


In [ ]:
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display
from utils.capture_feature_profile import run_capture_feature_profile

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
PACKET_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml"
PREPROCESSING_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_preprocessing_v1.yaml"
PREPARED_RUN_DIR = (
    DRIVE_ROOT / "prepared_runs" / "20260917T235058_827743Z_prepare_full_dev"
)
BATCH_SIZE = 250_000

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_feature_profile"
DRIVE_RUN_DIR = DRIVE_ROOT / "feature_profile_runs" / RUN_ID

print(f"Prepared input: {PREPARED_RUN_DIR}")
print(f"Profile output: {DRIVE_RUN_DIR}")


## 4. Run and persist the profile


In [ ]:
PROFILE = run_capture_feature_profile(
    manifest_path=MANIFEST_PATH,
    packet_schema_path=PACKET_SCHEMA_PATH,
    preprocessing_schema_path=PREPROCESSING_SCHEMA_PATH,
    prepared_run_dir=PREPARED_RUN_DIR,
    output_dir=DRIVE_RUN_DIR,
    batch_size=BATCH_SIZE,
)
print(f"Feature profile saved to {DRIVE_RUN_DIR / 'capture_feature_profile.json'}")


## 5. Review the profile

The validation-coverage tables are diagnostic only. Validation values are never added to a training encoder.


In [ ]:
scenario_rows = []
protocol_rows = []
port_rows = []
top_port_rows = []
categorical_rows = []
numeric_rows = []

numeric_features = [
    "frame_length", "ipv4_fragment_offset", "ipv4_length", "ipv4_ttl",
    "ssh_padding_length", "tcp_header_length", "tcp_payload_length",
    "tcp_window_value",
]
categorical_features = [
    "ethernet_type", "ipv4_dscp", "mqtt_connack_reason_code",
    "mqtt_message_type", "mqtt_qos", "mqtt_reserved_flag",
    "mqtt_subscription_qos", "mqtt_version", "ssh_direction",
]

for scenario, profile in PROFILE["scenario_profiles"].items():
    scenario_rows.append({"scenario": scenario, "packets": profile["packets"]})
    protocol_rows.append({"scenario": scenario, **profile["protocol_indicator_counts"]})
    for direction, counts in profile["port_role_counts"].items():
        for role, count in counts.items():
            port_rows.append({
                "scenario": scenario, "direction": direction, "role": role,
                "packets": count, "fraction": count / profile["packets"],
            })
    for direction, counts in profile["top_raw_nonnull_ports"].items():
        top_port_rows.append({
            "scenario": scenario, "direction": direction,
            "top_nonnull_ports": counts,
        })
    for feature in categorical_features:
        stats = profile["feature_statistics"][feature]
        categorical_rows.append({
            "scenario": scenario, "feature": feature,
            "nonnull": stats["nonnull"],
            "null_fraction": stats["null"] / profile["packets"],
            "distinct_nonnull": stats["distinct_nonnull"],
            "value_counts": profile["categorical_counts"][feature],
        })
    for feature in numeric_features:
        stats = profile["feature_statistics"][feature]
        numeric_rows.append({
            "scenario": scenario, "feature": feature,
            "nonnull": stats["nonnull"], "null": stats["null"],
            "minimum": stats["minimum"], "q01": stats["quantiles"]["0.01"],
            "median": stats["quantiles"]["0.5"],
            "q99": stats["quantiles"]["0.99"], "maximum": stats["maximum"],
        })

print("Candidate dimensions")
display(pd.DataFrame([PROFILE["candidate_dimensions"]]))
print("Scenario rows")
display(pd.DataFrame(scenario_rows))
print("Protocol indicator counts")
display(pd.DataFrame(protocol_rows))
print("Fixed TCP-port role distributions")
display(pd.DataFrame(port_rows))
print("Most frequent raw non-null TCP ports")
display(pd.DataFrame(top_port_rows))
print("Categorical-code profiles")
display(pd.DataFrame(categorical_rows))
print("Numeric-magnitude profiles")
display(pd.DataFrame(numeric_rows))

fold_rows = []
coverage_rows = []
for fold, profile in PROFILE["fold_profiles"].items():
    fold_rows.append({
        "fold": fold, "training_packets": profile["training_packets"],
        "training_constants": profile["training_constants"],
        "training_all_missing": profile["training_all_missing"],
    })
    for feature, coverage in profile["categorical_validation_coverage"].items():
        coverage_rows.append({"fold": fold, "feature": feature, **coverage})
print("Fold-training constants")
display(pd.DataFrame(fold_rows))
print("Validation categorical coverage without refitting")
display(pd.DataFrame(coverage_rows))


## 6. Stop for review

Do not freeze preprocessing or train XGB-P yet. Save the notebook with its outputs and review Section 5 together with `capture_feature_profile.json`.
